# Notebook 2: Data Cleaning

## Purpose

In notebook 1 I profiled the raw dataset and documented every data quality issue I found — wrong data types, missing values, business rule violations, duplicate rows, and inconsistent categorical labels. This notebook is where I fix all of those issues in a deliberate, logged sequence.

I work through ten steps in a specific order because the sequence matters. I remove duplicates before imputing missing values so I do not inflate the training signal from near-identical rows. I fix data types before running KNN imputation because the imputer needs proper numeric arrays. I correct business-logic violations after imputation so that records corrected by the imputer are also checked for validity.

Every operation is recorded in a `cleaning_log` dictionary that I save as `cleaning_metadata.json`. This gives me a full audit trail — I can see exactly what was changed, by how much, and in what order. The final output of this notebook is `cleaned_data.csv`, which becomes the input for notebook 3 (feature engineering).

**Steps in this notebook:**
1. Import libraries and initialise the cleaning log
2. Load raw data
3. Remove duplicate rows
4. Fix data types — date and time columns
5. Fix data types — numerical columns
6. Identify and apply missing value imputation strategy
7. Correct business logic violations (age, ratings, amounts, year column)
8. Standardise categorical text formatting
9. Create derived time-based columns
10. Final validation and export

## Step 1: Import Libraries and Initialise Cleaning Log

I bring in only what I need for this notebook. `pandas` and `numpy` handle all dataframe operations. From `sklearn.impute` I import both `SimpleImputer` (for categorical columns using mode) and `KNNImputer` (for numerical columns). I chose KNN over simple median imputation for `Age`, `Income`, `Amount`, and `Ratings` because KNN uses the relationships between columns when filling gaps — for example, if a customer's `Age` is missing, KNN will look at customers with similar `Income`, `Total_Purchases`, and other numerical features to estimate a realistic value, rather than just substituting the dataset-wide median which ignores all other available information.

I initialise `cleaning_log` as a dictionary here at the very start, before any cleaning operation runs. Every step below appends a structured record to `cleaning_log['operations']`. At the end of the notebook I serialise this to `cleaning_metadata.json` so there is a full, auditable trail of exactly what changed and in what order.

In [1]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from sklearn.impute import SimpleImputer, KNNImputer  # SimpleImputer for categoricals, KNNImputer for numericals
import warnings
warnings.filterwarnings('ignore')  # suppress sklearn convergence and pandas dtype warnings during imputation

## Step 2: Load Raw Data

I load the original raw file directly — not the profiling outputs from notebook 1. This ensures that the cleaning notebook is fully self-contained and can be re-run from scratch at any point without depending on intermediate state from notebook 1. I immediately make a copy `df = df_raw.copy()` so that `df_raw` is always available as an unmodified reference in case I need to diagnose an issue mid-notebook without having to reload from disk.

I print the initial shape to confirm the file loaded correctly before doing anything else. If the shape does not match what I saw in notebook 1, that is a signal to stop immediately and investigate.

In [2]:
df_raw = pd.read_csv('../data/new_retail_data.csv')
df = df_raw.copy()  # keep df_raw as an unmodified backup; all cleaning operations work on df
print(f"Initial Shape: {df.shape}")

Initial Shape: (302010, 30)


In [3]:
# cleaning_log will grow as each step appends a record — serialised to JSON at the end
cleaning_log = {
    'timestamp': datetime.now().isoformat(),
    'initial_shape': df.shape,
    'operations': []  # each cleaning operation appends a dict here for full audit trail
}

## Step 3: Remove Duplicate Rows

In notebook 1, I found duplicate rows in the dataset. I need to deal with these first, before any imputation or type-fixing, because KNN imputation will incorporate duplicate records into the distance calculation. If I impute first and remove duplicates second, I risk building my imputation model on artificially inflated data.

I use `keep='first'` so that when rows are identical, I retain the earliest occurrence and drop all subsequent ones. After removal I call `reset_index(drop=True)` to ensure the index is a clean sequence from 0 — this matters later when I use positional operations and when I merge results back into the dataframe.

In [4]:
initial_count = len(df)
duplicates_mask = df.duplicated(keep='first')  # True for every row that is an exact copy of an earlier row
duplicates_count = duplicates_mask.sum()
print(f"Duplicated rows found: {duplicates_count}")

Duplicated rows found: 4


In [5]:
df = df[~duplicates_mask].reset_index(drop=True)  # invert mask to keep non-duplicates; reset index to clean sequence

cleaning_log['operations'].append({
    'step': 'remove_duplicates',
    'rows_removed': int(duplicates_count),
    'rows_remaining': len(df)
})
print(f"Rows after duplicate removal: {len(df)}")

Rows after duplicate removal: 302006


## Step 4: Fix Data Types — Date and Time Columns

In notebook 1, I saw that `Date` was read as an `object` (string) instead of `datetime64`. This means pandas cannot perform any date arithmetic on it — no day-of-week extraction, no time-between-purchases calculation, nothing. I fix this now before any missing value analysis so that when I check for nulls in the next step, NaT values from failed date parsing show up correctly rather than being hidden as non-null strings.

I use `errors='coerce'` intentionally. Any value that cannot be parsed as a valid date becomes `NaT` rather than raising an exception. I then count those `NaT` values to see exactly how many dates were unparseable — that count goes into the cleaning log.

I also extract `Hour` from the `Time` column and create a combined `Transaction_DateTime` column. This combined timestamp is useful later for the demand forecasting model which needs a single chronological index rather than split date and time fields.

In [6]:
df["Date"] = pd.to_datetime(df['Date'], errors='coerce')  # errors='coerce' turns unparseable strings into NaT rather than raising
invalid_dates = df['Date'].isnull().sum()  # count NaT values produced by failed parses
print(f"Invalid dates found: {invalid_dates}")

Invalid dates found: 359


In [7]:
# Year came in as float64 because of missing values; convert to numeric first then fill from Date
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Year'] = df['Year'].fillna(df['Date'].dt.year)  # use the parsed Date column as source of truth for missing years
df['Year'] = df['Year'].astype('int64')

# Month stays as string — it is used as a label in reports, not as a numeric value
df['Month'] = df['Month'].astype(str)

In [8]:
# Parse Time column to a time object; format='%H:%M:%S' matches the HH:MM:SS strings in the raw data
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce').dt.time

# Combine Date and Time into a single sortable timestamp — needed by the demand forecasting LSTM model
df['Transaction_DateTime'] = pd.to_datetime(
    df['Date'].astype(str) + ' ' + df['Time'].astype(str),
    errors='coerce'
)

cleaning_log['operations'].append({
    'step': 'convert_datetime',
    'invalid_dates': int(invalid_dates),
    'method': 'pd.to_datetime with error coercion'
})

## Step 5: Fix Data Types — Numerical Columns

From the profiling in notebook 1, I know that `Customer_ID` and `Transaction_ID` came in as `float64`. This is a common pandas behaviour when a column contains any missing values — pandas cannot represent `NaN` in an integer column by default, so it silently upcasts to float. I fix this to `Int64` (the nullable integer type) which correctly handles both integers and missing values.

I also convert `Phone` and `Zipcode` to strings rather than integers. These are identifiers — performing arithmetic on them would be meaningless, and storing them as integers would silently drop any leading zeros.

Amount columns (`Amount`, `Total_Amount`, `Total_Purchases`) and `Ratings` go to `float64`. These will be the targets and features in the ML models, so they must be numeric before imputation.

In [9]:
# ID columns were float64 because of missing values; Int64 (capital I) is pandas nullable integer — handles NaN correctly
id_columns = ['Transaction_ID', 'Customer_ID']
for col in id_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].astype('Int64')  # Int64 not int64 — the nullable version supports missing values

# Phone and Zipcode are identifiers, not numbers — store as strings to preserve leading zeros
df['Phone'] = df['Phone'].astype('Int64').astype(str)
df['Zipcode'] = df['Zipcode'].astype('Int64').astype(str)

In [10]:
# Age may have arrived as float due to nulls; convert to nullable integer
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Age'] = df['Age'].astype('Int64')

In [11]:
# Amount columns must be float64 for KNN imputation and model training — int would lose decimal precision
amount_columns = ['Amount', 'Total_Amount', 'Total_Purchases']
for col in amount_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Ratings is float because values like 4.5 are valid — keeping as float preserves that granularity
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')

cleaning_log['operations'].append({
    'step': 'convert_numerical_types',
    'columns_converted': id_columns + amount_columns + ['Age', 'Income', 'Ratings']
})

## Step 6: Identify Missing Value Imputation Strategy

Before I run any imputation, I calculate the percentage of missing values per column and classify each column into one of four strategies:

- **Drop row** — for critical identifier columns (`Customer_ID`, `Transaction_ID`, `Date`). If any of these are missing, the entire row is unusable and I cannot reliably recover it.
- **Most frequent (mode)** — for categorical columns like `Gender`, `City`, `Country`. I fill with the most common value because for low-to-medium cardinality categoricals this is a safe, interpretable choice.
- **KNN (k=5)** — for numerical value columns: `Age`, `Income`, `Amount`, `Total_Amount`, `Ratings`. KNN preserves the correlation structure between these columns rather than using a single central statistic that ignores all other available information.
- **Constant "Unknown"** — for high-cardinality identifier strings like `Name`, `Email`, `Phone`. There is no sensible way to guess these, so I flag them explicitly as unknown rather than leaving them null or inventing values.

I define this as a function `get_missing_strategy()` so the logic is explicit, testable, and easy to modify if I later discover new column types.

In [12]:
# Build a summary table of missing values sorted by worst columns first
# This tells me which columns need imputation and how severe the gap is
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Pct': (df.isnull().sum() / len(df) * 100).round(2).values
})
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values(
    'Missing_Pct', ascending=False  # sort worst first so I can spot high-risk columns immediately
)

print(missing_summary.to_string(index=False))

              Column  Missing_Count  Missing_Pct
Transaction_DateTime            708         0.23
                Name            382         0.13
              Amount            356         0.12
                Time            350         0.12
     Total_Purchases            361         0.12
                Date            359         0.12
        Total_Amount            350         0.12
      Transaction_ID            333         0.11
               Email            347         0.11
     Shipping_Method            337         0.11
              Income            290         0.10
      Payment_Method            297         0.10
              Gender            317         0.10
             Address            315         0.10
         Customer_ID            308         0.10
             Country            271         0.09
               State            281         0.09
       Product_Brand            281         0.09
    Product_Category            283         0.09
        Order_Status

In [13]:
def get_missing_strategy(col_name, missing_pct, dtype):
    """
    Returns the imputation strategy for a column based on its name and type.
    I define this as a function rather than a hard-coded dict so the logic is
    explicit, readable, and easy to extend if new column types appear.
    """
    # Critical identifier columns — a row without these cannot be used at all
    if col_name in ['Customer_ID', 'Date', 'Transaction_ID']:
        return 'drop'

    # High-cardinality string identifiers — no sensible way to guess these
    elif col_name in ['Name', 'Email', 'Phone', 'Address']:
        return 'constant:Unknown'

    # Categorical columns — mode (most frequent) is a safe, interpretable fill
    elif dtype == 'object' or col_name in ['Gender', 'City', 'State', 'Country']:
        return 'most_frequent'

    # Numerical value columns — KNN uses inter-column correlations for better estimates
    elif col_name in ['Age', 'Income', 'Amount', 'Total_Amount', 'Ratings']:
        return 'knn'

    # Everything else — flag as unknown rather than inventing a value
    else:
        return 'constant:Unknown'

In [14]:
# Apply the strategy function to every column that has missing values
# Printing the result gives a readable summary of what will happen to each column
missing_strategy = {}
for _, row in missing_summary.iterrows():
    col = row['Column']
    strategy = get_missing_strategy(col, row['Missing_Pct'], df[col].dtype)
    missing_strategy[col] = strategy
    print(f"{col:25s} -> {strategy}")

Transaction_DateTime      -> constant:Unknown
Name                      -> constant:Unknown
Amount                    -> knn
Time                      -> most_frequent
Total_Purchases           -> constant:Unknown
Date                      -> drop
Total_Amount              -> knn
Transaction_ID            -> drop
Email                     -> constant:Unknown
Shipping_Method           -> most_frequent
Income                    -> most_frequent
Payment_Method            -> most_frequent
Gender                    -> most_frequent
Address                   -> constant:Unknown
Customer_ID               -> drop
Country                   -> most_frequent
State                     -> most_frequent
Product_Brand             -> most_frequent
Product_Category          -> most_frequent
Order_Status              -> most_frequent
City                      -> most_frequent
Customer_Segment          -> most_frequent
Age                       -> knn
Feedback                  -> most_frequent
Ratings   

## Step 7: Execute Imputation Strategy

Now I apply the four strategies I defined above, in the correct order:

1. **Drop rows for critical columns first** — I remove any rows where `Customer_ID`, `Transaction_ID`, or `Date` is null. Doing this first shrinks the dataset to only salvageable rows before I spend time imputing.
2. **Mode imputation for categoricals** — `SimpleImputer(strategy='most_frequent')` handles `Gender`, `City`, `Country`, and similar columns. This fills missing values with the most common observed value, which is appropriate for low-cardinality categoricals.
3. **KNN imputation for numericals** — `KNNImputer(n_neighbors=5)` fills `Age`, `Income`, `Amount`, `Total_Amount`, and `Ratings`. The k=5 setting means each missing value is estimated by averaging the 5 nearest neighbours in terms of all other numeric features.
4. **Constant fill for identifiers** — `Name`, `Email`, `Phone`, and similar columns with no recoverable value get filled with the literal string "Unknown".

After all four strategies run, I verify that `df.isnull().sum().sum()` equals zero. If any nulls remain at this point, that means I missed a column in the strategy function — which would require investigation before proceeding.

In [15]:
# Strategy 1: DROP rows where critical identifier columns are null
# A transaction with no Customer_ID, Transaction_ID, or Date cannot be used in any model
critical_cols = [k for k, v in missing_strategy.items() if v == 'drop']
if critical_cols:
    rows_before = len(df)
    df = df.dropna(subset=critical_cols).reset_index(drop=True)
    rows_after = len(df)
    print(f"Dropped {rows_before - rows_after} rows with missing critical columns: {critical_cols}")

    cleaning_log['operations'].append({
        'step': 'drop_critical_missing',
        'critical_cols': critical_cols,
        'rows_before': rows_before,
        'rows_after': rows_after,
        'rows_removed': rows_before - rows_after
    })

Dropped 1000 rows with missing critical columns: ['Date', 'Transaction_ID', 'Customer_ID']


In [16]:
# Strategy 2: Mode imputation for categorical columns
# SimpleImputer with strategy='most_frequent' fills each column with its own mode value
mode_cols = [k for k, v in missing_strategy.items() if v == 'most_frequent']
if mode_cols:
    imputer_mode = SimpleImputer(strategy='most_frequent')
    df[mode_cols] = imputer_mode.fit_transform(df[mode_cols])
    print(f"Filled most_frequent for columns: {mode_cols}")

Filled most_frequent for columns: ['Time', 'Shipping_Method', 'Income', 'Payment_Method', 'Gender', 'Country', 'State', 'Product_Brand', 'Product_Category', 'Order_Status', 'City', 'Customer_Segment', 'Feedback']


In [17]:
# Strategy 3: KNN imputation for numerical value columns
# n_neighbors=5 means each missing value is estimated from the 5 most similar rows
# "Similar" is measured using all other numerical features — so Age benefits from Income, Amount, etc.
knn_cols = [k for k, v in missing_strategy.items() if v == 'knn']
if knn_cols:
    imputer_knn = KNNImputer(n_neighbors=5)
    df[knn_cols] = imputer_knn.fit_transform(df[knn_cols])
    print(f"Filled KNN for columns: {knn_cols}")

Filled KNN for columns: ['Amount', 'Total_Amount', 'Age', 'Ratings']


In [18]:
# Strategy 4: Constant fill for high-cardinality identifiers
# These cannot be guessed meaningfully — flagging as "Unknown" is more honest than inventing a value
constant_cols = [k for k, v in missing_strategy.items() if v.startswith('constant:')]
if constant_cols:
    fill_value = missing_strategy[constant_cols[0]].split(':', 1)[1]  # parse fill value from "constant:Unknown"
    imputer_constant = SimpleImputer(strategy='constant', fill_value=fill_value)
    df[constant_cols] = imputer_constant.fit_transform(df[constant_cols])
    print(f"Filled '{fill_value}' for columns: {constant_cols}")

Filled 'Unknown' for columns: ['Transaction_DateTime', 'Name', 'Total_Purchases', 'Email', 'Address']


In [19]:
# Log what was imputed for the audit trail
cleaning_log['operations'].append({
    'step': 'impute_missing_values',
    'dropped_cols': critical_cols if 'critical_cols' in locals() else [],
    'most_frequent_cols': mode_cols if 'mode_cols' in locals() else [],
    'knn_cols': knn_cols if 'knn_cols' in locals() else [],
    'constant_cols': constant_cols if 'constant_cols' in locals() else []
})

# Verify that all missing values have been handled — if this is not zero something was missed
remaining_missing = df.isnull().sum().sum()
print(f"Remaining missing values after imputation: {remaining_missing}")

if remaining_missing > 0:
    # If this block runs it means get_missing_strategy() missed a column type
    print("Columns still with missing values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

Remaining missing values after imputation: 0


## Step 8: Correct Business Logic Violations

Even after imputation, some values may be outside acceptable real-world ranges. The imputer optimises for statistical consistency but does not know business constraints. I need to enforce four rules that I identified in notebook 1:

1. **Age must be 18–100** — I cap values below 18 at 18 and values above 100 at 100. These are hard limits for a retail customer dataset (under-18 transactions would be unusual and over-100 is statistically implausible). I cap rather than drop because capping preserves the row while correcting the value.
2. **Ratings must be 1–5** — The rating scale in this dataset is 1 to 5 inclusive. Values outside this range are almost certainly data entry errors. I cap them to the nearest valid boundary.
3. **Amounts must be non-negative** — Negative transaction amounts could represent refunds, but in this dataset the `Order_Status` column handles refunds. Negative `Amount` values are almost certainly sign errors — I take the absolute value.
4. **Year column must match the parsed Date** — In notebook 1, I found cases where the `Year` column did not match the year component of the `Date` column. The `Date` column is the authoritative source because it contains the full date information. Where there is a mismatch, I overwrite `Year` with `Date.dt.year`.

In [20]:
# Age validation: must be 18–100 for a retail customer dataset
# KNN may have produced values slightly outside this range due to averaging — cap rather than drop
invalid_age_mask = (df['Age'] < 18) | (df['Age'] > 100)
invalid_age_count = invalid_age_mask.sum()
print(f"Invalid age values: {invalid_age_count}")

if invalid_age_count > 0:
    df.loc[df['Age'] < 18, 'Age'] = 18   # floor at minimum adult age
    df.loc[df['Age'] > 100, 'Age'] = 100  # cap at max plausible age
    print(f"Corrected {invalid_age_count} invalid age values")

Invalid age values: 0


In [21]:
# Ratings validation: the dataset uses a 1–5 scale
# Values outside this range are almost certainly data entry errors — cap to nearest valid boundary
invalid_ratings_mask = (df['Ratings'] < 1) | (df['Ratings'] > 5)
invalid_ratings_count = invalid_ratings_mask.sum()
print(f"Invalid ratings: {invalid_ratings_count}")

if invalid_ratings_count > 0:
    df.loc[df['Ratings'] < 1, 'Ratings'] = 1.0  # floor at minimum valid rating
    df.loc[df['Ratings'] > 5, 'Ratings'] = 5.0  # cap at maximum valid rating
    print(f"Corrected {invalid_ratings_count} invalid rating values")

Invalid ratings: 0


In [22]:
# Negative amount check: genuine refunds are captured in Order_Status, not via negative values
# Any negative Amount or Total_Amount in this dataset is a sign-flip error — correct with abs()
negative_amount_mask = (df['Amount'] < 0) | (df['Total_Amount'] < 0)
negative_amount_count = negative_amount_mask.sum()
print(f"Negative amounts found: {negative_amount_count}")

if negative_amount_count > 0:
    df['Amount'] = df['Amount'].abs()        # flip negative to positive
    df['Total_Amount'] = df['Total_Amount'].abs()
    print(f"Corrected {negative_amount_count} negative amount values")

Negative amounts found: 0


In [23]:
# Year consistency check: the Year column should always equal the year component of the Date column
# I extract Date_Year as a temporary column for comparison, then drop it after correcting mismatches
df['Date_Year'] = df['Date'].dt.year
year_mismatch_mask = (df['Year'] != df['Date_Year']) & df['Date_Year'].notna()
year_mismatch_count = year_mismatch_mask.sum()
print(f"Year-Date mismatches: {year_mismatch_count}")

if year_mismatch_count > 0:
    # Date is the authoritative source — it contains the full parsed datetime
    df.loc[year_mismatch_mask, 'Year'] = df.loc[year_mismatch_mask, 'Date_Year']
    print(f"Corrected {year_mismatch_count} year-date mismatches")

df = df.drop('Date_Year', axis=1)  # remove temporary helper column

cleaning_log['operations'].append({
    'step': 'data_validation_corrections',
    'invalid_ages_corrected': int(invalid_age_count),
    'invalid_ratings_corrected': int(invalid_ratings_count),
    'negative_amounts_corrected': int(negative_amount_count),
    'year_mismatches_corrected': int(year_mismatch_count)
})

Year-Date mismatches: 0


## Step 9: Standardise Categorical Text Formatting

In notebook 1 I found inconsistent casing in categorical columns — values like `"male"`, `"Male"`, `"MALE"` all exist in `Gender`. When I later encode these columns for the ML models, each distinct string becomes its own category. Without standardisation, a single-category column like `Gender` could fragment into six or more categories purely because of case differences, which would corrupt the model.

I apply a consistent formatting rule to each category column:
- `Gender`, `Country`, `Customer_Segment`, `Order_Status`, `Payment_Method` — I use `.str.strip().str.title()` to remove whitespace and capitalise each word (e.g. "MALE" becomes "Male", "credit card" becomes "Credit Card")
- `State` — I use `.str.strip().str.upper()` because two-letter US state abbreviations are conventionally uppercase ("ca" should be "CA")

I print the unique values for each column after standardising so I can visually confirm the normalisation worked and spot any remaining anomalies.

In [24]:
if 'Gender' in df.columns:
    # .str.strip() removes leading/trailing whitespace; .str.title() normalises "MALE", "male" -> "Male"
    df['Gender'] = df['Gender'].str.strip().str.title()
    print(f"Gender values: {df['Gender'].unique()}")

Gender values: ['Male' 'Female']


In [25]:
if 'Country' in df.columns:
    df['Country'] = df['Country'].str.strip().str.title()  # normalise "united states", "UNITED STATES" -> "United States"
    print(f"Unique countries: {df['Country'].unique()}")

Unique countries: ['Germany' 'Uk' 'Australia' 'Canada' 'Usa']


In [26]:
if 'State' in df.columns:
    # Use uppercase for state codes — two-letter US abbreviations are conventionally uppercase (CA, NY, TX)
    df['State'] = df['State'].str.strip().str.upper()

In [27]:
if 'Customer_Segment' in df.columns:
    # Title case normalises "premium", "PREMIUM", "Premium" all to "Premium"
    df['Customer_Segment'] = df['Customer_Segment'].str.strip().str.title()
    print(f"Customer segments: {df['Customer_Segment'].unique()}")

Customer segments: ['Regular' 'Premium' 'New']


In [28]:
if 'Order_Status' in df.columns:
    df['Order_Status'] = df['Order_Status'].str.strip().str.title()  # "completed", "COMPLETED" -> "Completed"
    print(f"Order statuses: {df['Order_Status'].unique()}")

Order statuses: ['Shipped' 'Processing' 'Pending' 'Delivered']


In [29]:
if 'Payment_Method' in df.columns:
    # "credit card", "Credit Card", "CREDIT CARD" all become "Credit Card"
    df['Payment_Method'] = df['Payment_Method'].str.strip().str.title()
    print(f"Payment methods: {df['Payment_Method'].unique()}")

cleaning_log['operations'].append({
    'step': 'standardize_categorical',
    'columns': ['Gender', 'Country', 'State', 'Customer_Segment', 'Order_Status', 'Payment_Method']
})

Payment methods: ['Debit Card' 'Credit Card' 'Paypal' 'Cash']


## Step 10: Create Derived Time-Based Columns

I create these derived columns here in the cleaning notebook rather than in notebook 3 (feature engineering) because they are direct extractions from `Date` and `Time`. They require no modelling assumptions or domain knowledge — they are deterministic transformations that simply decompose the datetime into its components. Any column that is a mechanical extraction I treat as part of cleaning; any column that requires a decision or aggregation logic I put in feature engineering.

The columns I create:
- `DayOfWeek` — the calendar day name (Monday through Sunday); I use this in the EDA and in the demand forecasting model
- `DayOfMonth` — useful for detecting end-of-month spending patterns
- `Quarter` — aggregates months into Q1–Q4 for seasonal analysis  
- `IsWeekend` — binary flag (1 = Saturday or Sunday); this is one of the most predictive features for transaction volume in retail datasets
- `Hour` — extracted from the `Time` column; purchasing behaviour varies sharply by time of day
- `TimeOfDay` — categorical buckets (Night / Morning / Afternoon / Evening) derived from `Hour`; this is a human-readable version of the hour for EDA visualisations
- `DaysSinceFirstPurchase` — computed per customer; this is a direct input to the churn and CLV models because it measures customer tenure

In [30]:
# Extract date components — these are deterministic extractions, not engineered features
df['DayOfWeek'] = df['Date'].dt.day_name()   # full day name (Monday, Tuesday...) for EDA visualisations
df['DayOfMonth'] = df['Date'].dt.day         # 1–31; useful for detecting end-of-month spending spikes
df['Quarter'] = df['Date'].dt.quarter        # 1–4; useful for seasonal trend analysis
df['IsWeekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)  # 1 = Sat/Sun; strong predictor in retail

In [31]:
# Extract hour from the Time column — purchasing behaviour varies sharply by hour of day
# Re-parse the Time object to datetime so .dt.hour works cleanly
df['Hour'] = pd.to_datetime(df['Time'].astype(str), format='%H:%M:%S', errors='coerce').dt.hour

In [32]:
def categorize_time_of_day(hour):
    """
    Converts a 24-hour integer to a readable time-of-day label.
    I use this categorical version in EDA charts — 'Morning' is easier to read than '9'
    when presenting patterns to a non-technical audience.
    """
    if pd.isna(hour):
        return 'Unknown'
    elif hour < 6:
        return 'Night'
    elif hour < 12:
        return 'Morning'
    elif hour < 17:
        return 'Afternoon'
    elif hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['TimeOfDay'] = df['Hour'].apply(categorize_time_of_day)

In [33]:
# Calculate days since each customer's first transaction — a direct measure of customer tenure
# I sort by Customer_ID and Date first so that groupby transform correctly identifies the first purchase date
df_sorted = df.sort_values(['Customer_ID', 'Date'])

# groupby().transform('min') computes the minimum Date per customer and broadcasts it back to every row
df_sorted['CustomerFirstPurchase'] = df_sorted.groupby('Customer_ID')['Date'].transform('min')

# Subtract first purchase date from each transaction date to get tenure in days
df_sorted['DaysSinceFirstPurchase'] = (df_sorted['Date'] - df_sorted['CustomerFirstPurchase']).dt.days

df = df_sorted  # replace df with the sorted version — order is preserved for downstream models

cleaning_log['operations'].append({
    'step': 'create_derived_columns',
    'new_columns': ['DayOfWeek', 'DayOfMonth', 'Quarter', 'IsWeekend', 'Hour', 'TimeOfDay', 'DaysSinceFirstPurchase']
})

## Step 11: Final Validation and Export

Before saving, I run a final quality check to confirm that the entire cleaning pipeline worked end-to-end. I check for any remaining `null` values, any remaining duplicates, and the final shape. I also compute a data quality score as the proportion of non-null cells — this is a simple but useful single number to include in any documentation or report.

I save three artefacts:
1. `data/cleaned_data.csv` — the cleaned dataframe that notebook 3 will load as its starting point
2. `data/cleaning_metadata.json` — the full `cleaning_log` dictionary with every operation, row count before and after, and column lists; this makes the cleaning process reproducible and auditable
3. `reports/cleaning_report.txt` — a plain-text summary for quick reference without needing to open the notebook

## Summary

After running this notebook, the dataset is clean and ready for feature engineering. The sequence of operations I performed:
- Removed duplicate rows
- Parsed dates and times to correct dtypes, creating a combined `Transaction_DateTime` column
- Coerced numerical and identifier columns to appropriate types (`Int64`, `float64`, `str`)
- Applied four-strategy imputation: drop for critical nulls, KNN for numerical, mode for categorical, constant for identifiers
- Corrected business logic violations in age, ratings, amounts, and the year column
- Standardised casing across all categorical columns
- Created seven derived time-based columns for use in downstream models

The output `cleaned_data.csv` has zero missing values, zero duplicates, and correct data types throughout. Notebook 3 picks up from here to engineer the model features.

In [34]:
# Final quality check — confirm all cleaning operations worked end-to-end before saving
print(f"Final Shape:    {df.shape}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicates:     {df.duplicated().sum()}")
print(f"Memory Usage:   {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Data quality score = proportion of non-null cells — a single summary metric
total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isnull().sum().sum()
quality_score = ((total_cells - missing_cells) / total_cells) * 100
print(f"\nData Quality Score: {quality_score:.2f}%")

# Finalise the cleaning log with output metadata
cleaning_log['final_shape'] = df.shape
cleaning_log['quality_score'] = float(quality_score)
cleaning_log['missing_values_remaining'] = int(df.isnull().sum().sum())

# Artefact 1: cleaned dataframe — input for notebook 3 (feature engineering)
output_path = '../data/cleaned_data.csv'
df.to_csv(output_path, index=False)
print(f"\nCleaned data saved to: {output_path}")

# Artefact 2: cleaning audit log — records every operation, row counts, and column lists
with open('../data/cleaning_metadata.json', 'w') as f:
    json.dump(cleaning_log, f, indent=2)

# Artefact 3: plain-text report — human-readable summary for documentation purposes
with open('../reports/cleaning_report.txt', 'w') as f:
    f.write("DATA CLEANING REPORT\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Timestamp: {cleaning_log['timestamp']}\n")
    f.write(f"Initial Shape: {cleaning_log['initial_shape']}\n")
    f.write(f"Final Shape: {cleaning_log['final_shape']}\n")
    f.write(f"Quality Score: {quality_score:.2f}%\n\n")
    f.write("Operations Performed:\n")
    for i, op in enumerate(cleaning_log['operations'], 1):
        f.write(f"{i}. {op['step']}\n")

Final Shape:    (301006, 39)
Missing Values: 0
Duplicates:     0
Memory Usage:   508.46 MB

Data Quality Score: 100.00%

Cleaned data saved to: ../data/cleaned_data.csv
